# LeetCode 3427 — Sum of Variable Length Subarrays bằng Python

Link đề: https://leetcode.com/problems/sum-of-variable-length-subarrays/ · Độ khó: **Easy** · Chủ đề: Array (mảng), Prefix Sum (tổng tiền tố)

Skeleton LeetCode:

```python
class Solution:
    def subarraySum(self, nums: List[int]) -> int:
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề; mọi đáp án trong notebook đã được kiểm chứng bằng code (so sánh năm cách làm với nhau và với một bộ vét cạn viết riêng). Ví dụ nào do mình tự dựng sẽ ghi rõ "tự dựng" — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: với **mỗi** chỉ số `i`, ta xét mảng con `nums[start ... i]` với `start = max(0, i - nums[i])`; cần **cộng tổng** của tất cả các mảng con đó (độ dài mảng con **thay đổi** theo giá trị `nums[i]`).
2. Nắm các công cụ Python: **cắt lát** `nums[l:r]` (nửa mở: lấy `l` đến `r - 1`), `sum`, `max(0, ...)` để chặn biên dưới, `enumerate`, `itertools.accumulate` (có tham số `initial=0`) và mảng hiệu (difference array).
3. Đi từ cách **cộng bằng vòng lặp trong** → **cắt lát + `sum`** → **tổng tiền tố (prefix sum) tự viết** → **`accumulate(initial=0)`** → **mảng hiệu đếm "mỗi phần tử được cộng bao nhiêu lần"**.
4. Hiểu mấu chốt: tổng một đoạn `nums[l..r]` bằng `s[r + 1] - s[l]` với `s` là tổng tiền tố, nên mỗi câu hỏi trả lời trong `O(1)` sau khi tiền xử lý `O(n)`.
5. Tránh các lỗi kinh điển: lệch một (off-by-one) giữa `s[i]` và `s[i + 1]`, quên `max(0, ...)` (chỉ số âm trong Python lại **không** báo lỗi mà cho kết quả **sai**), và cắt lát `nums[start:i]` thiếu phần tử `i`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` kích thước `n`. Với **mỗi** chỉ số `i` (`0 <= i < n`), xác định mảng con `nums[start ... i]` trong đó `start = max(0, i - nums[i])`.

Trả về **tổng** của tất cả các phần tử thuộc các mảng con vừa xác định cho từng chỉ số.

**Ví dụ 1**

- Input: `nums = [2,3,1]`
- Output: `11`
- Giải thích:

| `i` | mảng con | tổng |
|---|---|---|
| 0 | `nums[0] = [2]` | 2 |
| 1 | `nums[0 ... 1] = [2, 3]` | 5 |
| 2 | `nums[1 ... 2] = [3, 1]` | 4 |
| **Tổng** | | **11** |

**Ví dụ 2**

- Input: `nums = [3,1,1,2]`
- Output: `13`
- Giải thích:

| `i` | mảng con | tổng |
|---|---|---|
| 0 | `nums[0] = [3]` | 3 |
| 1 | `nums[0 ... 1] = [3, 1]` | 4 |
| 2 | `nums[1 ... 2] = [1, 1]` | 2 |
| 3 | `nums[1 ... 3] = [1, 1, 2]` | 4 |
| **Tổng** | | **13** |

**Ràng buộc**

- `1 <= n == nums.length <= 100`
- `1 <= nums[i] <= 1000`

In [1]:
from typing import List

examples = [
    ([2, 3, 1], 11),
    ([3, 1, 1, 2], 13),
]

for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [2, 3, 1] -> kỳ vọng 11
nums = [3, 1, 1, 2] -> kỳ vọng 13


## 3. Kiến thức nền

### 3.1. Cắt lát `nums[l:r]` là nửa mở

`nums[l:r]` lấy các phần tử có chỉ số `l, l+1, ..., r-1` — **không gồm `r`**. Vì đề muốn mảng con `nums[start ... i]` **gồm cả `i`**, ta phải cắt `nums[start : i + 1]`. Đây là nguồn lỗi lệch một số một.

In [2]:
nums = [10, 20, 30, 40, 50]
print(nums[1:3])          # [20, 30]  -> chỉ số 1, 2 (không có 3)
print(nums[1:3 + 1])      # [20, 30, 40]  -> chỉ số 1..3 gồm cả 3
print(nums[:2], nums[3:]) # [10, 20] [40, 50]

[20, 30]
[20, 30, 40]
[10, 20] [40, 50]


### 3.2. `sum` và `max(0, ...)`

`sum(list)` cộng mọi phần tử (`sum([]) = 0`). `max(0, x)` cho `x` nếu `x >= 0` và cho `0` nếu `x < 0` — tức "chặn dưới tại 0". Ta cần nó cho `start = max(0, i - nums[i])`: nếu `i - nums[i]` âm (mảng con muốn vươn quá đầu mảng) thì `start` dừng ở `0`.

In [3]:
print(sum([2, 3, 1]), sum([]))             # 6 0
for i, v in [(0, 2), (1, 3), (2, 1)]:
    print(f"i={i}, nums[i]={v}: i - nums[i] = {i - v}, start = max(0, ...) = {max(0, i - v)}")

6 0
i=0, nums[i]=2: i - nums[i] = -2, start = max(0, ...) = 0
i=1, nums[i]=3: i - nums[i] = -2, start = max(0, ...) = 0
i=2, nums[i]=1: i - nums[i] = 1, start = max(0, ...) = 1


### 3.3. Cạm bẫy: chỉ số âm trong Python **không** báo lỗi

Trong Python, `nums[-1]` là phần tử cuối, và `nums[-3:5]` bắt đầu cắt từ **vị trí thứ ba tính từ cuối**. Vì vậy nếu quên `max(0, ...)` mà đưa `i - nums[i]` âm vào phép cắt lát, chương trình **vẫn chạy** nhưng cho kết quả **khác hẳn** mong đợi. Lỗi kiểu "im lặng" như vậy rất khó phát hiện — hãy luôn chặn biên.

In [4]:
nums = [2, 3, 1]
i = 0
bad_start = i - nums[i]            # 0 - 2 = -2
print("start sai:", bad_start)
print("cắt sai  nums[-2:1] =", nums[bad_start:i + 1])       # [] -> rỗng! (từ -2 tức chỉ số 1 đến chỉ số 0)
print("cắt đúng nums[0:1]  =", nums[max(0, bad_start):i + 1])   # [2]

start sai: -2
cắt sai  nums[-2:1] = []
cắt đúng nums[0:1]  = [2]


### 3.4. `enumerate` để lấy cả chỉ số `i` lẫn giá trị `nums[i]`

Vì mỗi mảng con phụ thuộc vào **cả** chỉ số `i` **và** giá trị `nums[i]`, ta dùng `for i, x in enumerate(nums):`: `i` là chỉ số, `x` là `nums[i]`.

In [5]:
nums = [3, 1, 1, 2]
for i, x in enumerate(nums):
    start = max(0, i - x)
    print(f"i={i}, x={x}: mảng con nums[{start}..{i}] = {nums[start:i + 1]}")

i=0, x=3: mảng con nums[0..0] = [3]
i=1, x=1: mảng con nums[0..1] = [3, 1]
i=2, x=1: mảng con nums[1..2] = [1, 1]
i=3, x=2: mảng con nums[1..3] = [1, 1, 2]


### 3.5. Tổng tiền tố (prefix sum)

**Ý tưởng:** tạo mảng `s` sao cho `s[k]` = tổng `k` phần tử đầu tiên của `nums`: `s[0] = 0`, `s[1] = nums[0]`, `s[2] = nums[0] + nums[1]`, ... Khi đó tổng đoạn `nums[l..r]` (gồm cả hai đầu) là

`s[r + 1] - s[l]`.

Vì sao? `s[r + 1]` là tổng từ chỉ số `0` đến `r`; `s[l]` là tổng từ `0` đến `l - 1`; trừ nhau còn đúng đoạn `l..r`. Mảng `s` dài `n + 1` (phần tử `s[0] = 0` để xử lý đoạn bắt đầu từ chỉ số `0`).

In [6]:
nums = [2, 3, 1]
s = [0]
for x in nums:
    s.append(s[-1] + x)
print("s =", s)                                    # [0, 2, 5, 6]
l, r = 1, 2
print(f"tổng nums[{l}..{r}] = s[{r + 1}] - s[{l}] = {s[r + 1] - s[l]}   (kiểm tra: {sum(nums[l:r + 1])})")
l, r = 0, 1
print(f"tổng nums[{l}..{r}] = s[{r + 1}] - s[{l}] = {s[r + 1] - s[l]}   (kiểm tra: {sum(nums[l:r + 1])})")

s = [0, 2, 5, 6]
tổng nums[1..2] = s[3] - s[1] = 4   (kiểm tra: 4)
tổng nums[0..1] = s[2] - s[0] = 5   (kiểm tra: 5)


### 3.6. `itertools.accumulate` và `initial=0`

`accumulate(nums)` sinh các **tổng dồn** `nums[0]`, `nums[0] + nums[1]`, ... Tham số `initial=0` thêm một số `0` ở **đầu**, cho đúng mảng tiền tố `s` dài `n + 1` ở trên.

In [7]:
from itertools import accumulate

nums = [2, 3, 1]
print(list(accumulate(nums)))                  # [2, 5, 6]        (thiếu s[0] = 0)
print(list(accumulate(nums, initial=0)))       # [0, 2, 5, 6]     (đúng s)

[2, 5, 6]
[0, 2, 5, 6]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `nums = [3, 1, 1, 2]`. Với mỗi `i`, tính `start` rồi cộng tay các phần tử:

| `i` | `nums[i]` | `i - nums[i]` | `start` | mảng con | tổng |
|---|---|---|---|---|---|
| 0 | 3 | -3 | 0 | `[3]` | 3 |
| 1 | 1 | 0 | 0 | `[3, 1]` | 4 |
| 2 | 1 | 1 | 1 | `[1, 1]` | 2 |
| 3 | 2 | 1 | 1 | `[1, 1, 2]` | 4 |

Cộng các tổng: `3 + 4 + 2 + 4 = 13`.

Hai nhận xét: (1) mảng con luôn **kết thúc ở `i`** và chỉ "vươn về bên trái" `nums[i]` bước (hoặc đến đầu mảng); (2) các mảng con **chồng lên nhau** nhiều, nên cộng lại từng phần tử mỗi lần là lặp lại việc cũ — gợi ý dùng **tổng tiền tố**.

In [8]:
nums = [3, 1, 1, 2]
total = 0
for i, x in enumerate(nums):
    start = max(0, i - x)
    part = sum(nums[start:i + 1])
    total += part
    print(f"i={i}: start={start}, nums[{start}..{i}]={nums[start:i + 1]}, tổng={part}, cộng dồn={total}")

i=0: start=0, nums[0..0]=[3], tổng=3, cộng dồn=3
i=1: start=0, nums[0..1]=[3, 1], tổng=4, cộng dồn=7
i=2: start=1, nums[1..2]=[1, 1], tổng=2, cộng dồn=9
i=3: start=1, nums[1..3]=[1, 1, 2], tổng=4, cộng dồn=13


## 5. Cách 1 — Brute Force / baseline: vòng lặp trong cộng từng phần tử

**Ý tưởng.** Làm đúng như tính tay, nhưng **không dùng** cắt lát hay `sum`: với mỗi `i`, tính `start`, rồi dùng một vòng `for j in range(start, i + 1)` cộng `nums[j]` vào `total`.

In [9]:
def sum_variable_loops(nums: List[int]) -> int:
    total = 0
    for i in range(len(nums)):
        start = max(0, i - nums[i])
        for j in range(start, i + 1):            # i + 1 vì range không gồm điểm cuối
            total += nums[j]
    return total


for nums, expected in examples:
    print(sum_variable_loops(nums), "(kỳ vọng", expected, ")")

11 (kỳ vọng 11 )
13 (kỳ vọng 13 )


**Độ phức tạp (Cách 1).** Với mỗi `i`, vòng trong chạy tối đa `min(i, nums[i]) + 1` lần, tức tối đa `i + 1`. Tổng cộng khoảng `n(n+1)/2` bước ⇒ **Time `O(n²)`**, **Space `O(1)`**. Với `n <= 100` thì chỉ cỡ `5050` bước — vẫn rất nhanh, nhưng ý tưởng sẽ chậm nếu `n` lớn.

**Nút thắt:** mỗi `i` ta **cộng lại** những phần tử đã cộng ở các `i` trước. Tổng tiền tố cho phép lấy tổng bất kỳ đoạn nào chỉ bằng **một phép trừ**.

## 6. Ý tưởng thuật toán / bước đột phá

**Tính trước tổng tiền tố một lần, rồi mỗi mảng con chỉ tốn một phép trừ.**

Với `s` là mảng tiền tố (`s[0] = 0`, `s[k] = nums[0] + ... + nums[k-1]`):

`tổng của mảng con tại i = s[i + 1] - s[start]` với `start = max(0, i - nums[i])`.

Cộng cho mọi `i` là xong. Tiền xử lý `O(n)`, mỗi `i` tốn `O(1)` ⇒ tổng `O(n)`.

In [10]:
nums = [2, 3, 1]
s = list(accumulate(nums, initial=0))
print("s =", s)
total = 0
for i, x in enumerate(nums):
    start = max(0, i - x)
    part = s[i + 1] - s[start]
    total += part
    print(f"i={i}: s[{i + 1}] - s[{start}] = {s[i + 1]} - {s[start]} = {part}, tổng = {total}")

s = [0, 2, 5, 6]
i=0: s[1] - s[0] = 2 - 0 = 2, tổng = 2
i=1: s[2] - s[0] = 5 - 0 = 5, tổng = 7
i=2: s[3] - s[1] = 6 - 2 = 4, tổng = 11


## 7. Các cách tiếp theo

### Cách 2 — Cắt lát + `sum` (vẫn `O(n²)`, nhưng gọn)

**Giải thích.** `sum(nums[start : i + 1])` thay cho vòng lặp trong. Python thực hiện việc cộng bằng code C bên dưới nên nhanh hơn Cách 1 trong thực tế, nhưng **độ phức tạp tiệm cận không đổi**: mỗi lần cắt lát vẫn tạo list `O(độ dài)` rồi cộng `O(độ dài)`.

In [11]:
def sum_variable_slices(nums: List[int]) -> int:
    return sum(sum(nums[max(0, i - x): i + 1]) for i, x in enumerate(nums))


for nums, expected in examples:
    print(sum_variable_slices(nums), "(kỳ vọng", expected, ")")

11 (kỳ vọng 11 )
13 (kỳ vọng 13 )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionSlices:
    def subarraySum(self, nums: List[int]) -> int:
        total = 0
        for i, x in enumerate(nums):
            total += sum(nums[max(0, i - x): i + 1])
        return total


print(SolutionSlices().subarraySum([2, 3, 1]))          # 11

11


**Độ phức tạp Cách 2:** **Time `O(n²)`**, **Space `O(n)`** (list tạm do cắt lát). Chấp nhận được với `n <= 100`, và là cách nhiều người nộp đầu tiên.

### Cách 3 — Tổng tiền tố tự viết bằng vòng lặp

**Giải thích từng đoạn.**

1. `s = [0]`, rồi với mỗi `x` trong `nums` thêm `s[-1] + x`. Sau vòng lặp `len(s) = n + 1`.
2. Với mỗi `i`, `start = max(0, i - nums[i])`.
3. Cộng `s[i + 1] - s[start]` vào `total`.

In [13]:
def sum_variable_prefix_manual(nums: List[int]) -> int:
    s = [0]                                           # s[k] = tổng k phần tử đầu
    for x in nums:
        s.append(s[-1] + x)
    total = 0
    for i, x in enumerate(nums):
        start = max(0, i - x)
        total += s[i + 1] - s[start]                  # tổng nums[start..i]
    return total


for nums, expected in examples:
    print(sum_variable_prefix_manual(nums), "(kỳ vọng", expected, ")")

11 (kỳ vọng 11 )
13 (kỳ vọng 13 )


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionPrefixManual:
    def subarraySum(self, nums: List[int]) -> int:
        n = len(nums)
        s = [0] * (n + 1)
        for i in range(n):
            s[i + 1] = s[i] + nums[i]
        total = 0
        for i in range(n):
            total += s[i + 1] - s[max(0, i - nums[i])]
        return total


print(SolutionPrefixManual().subarraySum([3, 1, 1, 2]))        # 13

13


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** cho mảng `s`.

### Cách 4 — `accumulate(initial=0)` một dòng

**Giải thích.** Thay vòng lặp dựng `s` bằng `list(accumulate(nums, initial=0))`; phần còn lại là một biểu thức `sum(... for i, x in enumerate(nums))`. Đây chính là lời giải mẫu của đề.

In [15]:
def sum_variable_accumulate(nums: List[int]) -> int:
    s = list(accumulate(nums, initial=0))
    return sum(s[i + 1] - s[max(0, i - x)] for i, x in enumerate(nums))


for nums, expected in examples:
    print(sum_variable_accumulate(nums), "(kỳ vọng", expected, ")")

11 (kỳ vọng 11 )
13 (kỳ vọng 13 )


In [16]:
# Code nộp LeetCode cho Cách 4
from itertools import accumulate


class SolutionAccumulate:
    def subarraySum(self, nums: List[int]) -> int:
        s = list(accumulate(nums, initial=0))
        return sum(s[i + 1] - s[max(0, i - x)] for i, x in enumerate(nums))


print(SolutionAccumulate().subarraySum([2, 3, 1]))             # 11

11


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`**.

### Cách 5 — Mảng hiệu: đếm mỗi phần tử được cộng bao nhiêu lần

**Giải thích.** Đảo cách nhìn: thay vì "mỗi mảng con có tổng bao nhiêu", hỏi "**phần tử `nums[j]` nằm trong bao nhiêu mảng con**" — gọi là `cover[j]`. Khi đó kết quả là `Σ nums[j] × cover[j]`.

Để tính `cover` nhanh: mỗi mảng con `[start, i]` làm tăng `cover` thêm `1` trên cả đoạn. Dùng **mảng hiệu** `diff`: `diff[start] += 1`, `diff[i + 1] -= 1`; sau đó `cover` là tổng dồn của `diff`. Mỗi đoạn tốn `O(1)`.

In [17]:
def sum_variable_difference_array(nums: List[int]) -> int:
    n = len(nums)
    diff = [0] * (n + 1)                              # thêm 1 ô để ghi diff[i + 1] khi i = n - 1
    for i, x in enumerate(nums):
        start = max(0, i - x)
        diff[start] += 1                              # bắt đầu phủ từ start
        diff[i + 1] -= 1                              # hết phủ sau i
    cover = list(accumulate(diff))                    # cover[j] = số mảng con chứa nums[j]
    return sum(nums[j] * cover[j] for j in range(n))


for nums, expected in examples:
    print(sum_variable_difference_array(nums), "(kỳ vọng", expected, ")")

nums = [3, 1, 1, 2]
n = len(nums)
diff = [0] * (n + 1)
for i, x in enumerate(nums):
    diff[max(0, i - x)] += 1
    diff[i + 1] -= 1
print("diff  =", diff)
print("cover =", list(accumulate(diff))[:n], "-> mỗi phần tử nằm trong bao nhiêu mảng con")

11 (kỳ vọng 11 )
13 (kỳ vọng 13 )
diff  = [2, 1, -1, -1, -1]
cover = [2, 3, 2, 1] -> mỗi phần tử nằm trong bao nhiêu mảng con


In [18]:
# Code nộp LeetCode cho Cách 5
from itertools import accumulate


class SolutionDifferenceArray:
    def subarraySum(self, nums: List[int]) -> int:
        n = len(nums)
        diff = [0] * (n + 1)
        for i, x in enumerate(nums):
            diff[max(0, i - x)] += 1
            diff[i + 1] -= 1
        cover = list(accumulate(diff))
        return sum(nums[j] * cover[j] for j in range(n))


print(SolutionDifferenceArray().subarraySum([3, 1, 1, 2]))     # 13

13


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(n)`**. Kỹ thuật mảng hiệu rất hữu ích cho bài "cộng `+1` vào nhiều đoạn" (ví dụ LeetCode 370 Range Addition, 1109 Corporate Flight Bookings).

Bản nộp mình chọn là **Cách 4** (`accumulate`) — đúng ý tưởng cốt lõi (tổng tiền tố) và ngắn nhất; **Cách 3** là bản "tự viết" cùng ý, dễ hiểu nhất cho người mới.

## 8. TRACE từng bước

Trace **Cách 3** (tiền tố tự viết) với `nums = [3, 1, 1, 2]`. Trước hết mảng tiền tố: `s = [0, 3, 4, 5, 7]`.

| `i` | `nums[i]` | `start = max(0, i - nums[i])` | `s[i + 1]` | `s[start]` | phần = `s[i+1] - s[start]` | `total` |
|---|---|---|---|---|---|---|
| 0 | 3 | 0 | 3 | 0 | 3 | 3 |
| 1 | 1 | 0 | 4 | 0 | 4 | 7 |
| 2 | 1 | 1 | 5 | 3 | 2 | 9 |
| 3 | 2 | 1 | 7 | 3 | 4 | 13 |

Đáp án `13`.

In [19]:
def trace_prefix(nums: List[int]) -> int:
    s = [0]
    for x in nums:
        s.append(s[-1] + x)
    print(f"nums = {nums}")
    print(f"s    = {s}")
    total = 0
    for i, x in enumerate(nums):
        start = max(0, i - x)
        part = s[i + 1] - s[start]
        total += part
        print(f"  i={i} x={x}: start={start}, s[{i + 1}] - s[{start}] = {s[i + 1]} - {s[start]} = {part}, total = {total}")
    return total


assert trace_prefix([3, 1, 1, 2]) == 13

nums = [3, 1, 1, 2]
s    = [0, 3, 4, 5, 7]
  i=0 x=3: start=0, s[1] - s[0] = 3 - 0 = 3, total = 3
  i=1 x=1: start=0, s[2] - s[0] = 4 - 0 = 4, total = 7
  i=2 x=1: start=1, s[3] - s[1] = 5 - 3 = 2, total = 9
  i=3 x=2: start=1, s[4] - s[1] = 7 - 3 = 4, total = 13


Trace **Cách 5** (mảng hiệu) với `nums = [2, 3, 1]` để thấy mảng `diff` và `cover` thay đổi qua từng `i`:

In [20]:
def trace_difference(nums: List[int]) -> int:
    n = len(nums)
    diff = [0] * (n + 1)
    print(f"nums = {nums}")
    for i, x in enumerate(nums):
        start = max(0, i - x)
        diff[start] += 1
        diff[i + 1] -= 1
        print(f"  i={i} x={x}: đoạn [{start}, {i}] -> diff = {diff}")
    cover = list(accumulate(diff))[:n]
    total = sum(nums[j] * cover[j] for j in range(n))
    print("cover =", cover)
    print("tổng  =", " + ".join(f"{nums[j]}*{cover[j]}" for j in range(n)), "=", total)
    return total


assert trace_difference([2, 3, 1]) == 11

nums = [2, 3, 1]
  i=0 x=2: đoạn [0, 0] -> diff = [1, -1, 0, 0]
  i=1 x=3: đoạn [0, 1] -> diff = [2, -1, -1, 0]
  i=2 x=1: đoạn [1, 2] -> diff = [2, 0, -1, -1]
cover = [2, 2, 1]
tổng  = 2*2 + 3*2 + 1*1 = 11


## 9. Code nộp LeetCode

Bản nộp dùng Cách 4 (`accumulate(initial=0)`). `List` và `accumulate` — ở LeetCode `List` đã import sẵn, còn `accumulate` cần `from itertools import accumulate` nếu thiếu (nhiều trình chấm đã import sẵn, nhưng viết rõ cho chắc).

In [21]:
from itertools import accumulate


class Solution:
    def subarraySum(self, nums: List[int]) -> int:
        s = list(accumulate(nums, initial=0))                          # s[k] = tổng k phần tử đầu
        return sum(s[i + 1] - s[max(0, i - x)] for i, x in enumerate(nums))


sol = Solution()
for nums, expected in examples:
    print(sol.subarraySum(nums), "(kỳ vọng", expected, ")")

11 (kỳ vọng 11 )
13 (kỳ vọng 13 )


## 10. Vì sao thuật toán đúng

- **Công thức đoạn:** `s[k]` là tổng `k` phần tử đầu nên `s[i + 1] - s[start] = nums[start] + ... + nums[i]` đúng bằng tổng mảng con `nums[start..i]` (gồm cả hai đầu).
- **`s[0] = 0` rất quan trọng:** khi `start = 0`, ta cần `s[0] = 0` để phép trừ cho đúng tổng từ đầu mảng; vì vậy `initial=0`.
- **Chặn biên:** `start = max(0, i - nums[i])` đảm bảo `start >= 0` (đề cũng định nghĩa như vậy) nên chỉ số `s[start]` hợp lệ; và `start <= i` vì `nums[i] >= 1` nên `i - nums[i] <= i - 1 < i`.
- **Cộng mọi `i`:** vòng lặp `enumerate(nums)` đi qua mọi `i` đúng một lần, nên tổng cuối cùng là tổng mọi mảng con theo đề.
- **Cách 5 đồng nhất:** `Σ_i Σ_{j ∈ [start_i, i]} nums[j] = Σ_j nums[j] × (số i có j ∈ [start_i, i]) = Σ_j nums[j] × cover[j]` (đổi thứ tự lấy tổng).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (`n <= 100`).

| Cách | Time | Space |
|---|---|---|
| 1. Vòng lặp trong | `O(n²)` | `O(1)` |
| 2. Cắt lát + `sum` | `O(n²)` | `O(n)` |
| 3. Tiền tố tự viết | `O(n)` | `O(n)` |
| 4. `accumulate(initial=0)` | `O(n)` | `O(n)` |
| 5. Mảng hiệu + `cover` | `O(n)` | `O(n)` |

Vì `n <= 100`, mọi cách đều chạy tức thì. Tuy vậy, Cách 3–5 cho thấy kỹ thuật cần thiết khi `n` lớn (ví dụ `10^5`), nơi `O(n²)` sẽ quá chậm.

## 12. Bộ test

Gom các cách vào `impls` rồi so với kỳ vọng. "Trọng tài" `oracle` được viết theo cách khác: mô tả đúng từng chữ của đề bằng `for` lồng, không dùng prefix, không cắt lát.

In [22]:
import random

sol = Solution()
impls = {
    "loops": sum_variable_loops,
    "slices": sum_variable_slices,
    "prefix_manual": sum_variable_prefix_manual,
    "accumulate": sum_variable_accumulate,
    "difference_array": sum_variable_difference_array,
    "leetcode": sol.subarraySum,
}


def oracle(nums):
    """Trọng tài: duyệt từng i, từng j trong [start, i], cộng nums[j] (đúng từng chữ của đề)."""
    total = 0
    for i in range(len(nums)):
        start = i - nums[i]
        if start < 0:
            start = 0
        for j in range(start, i + 1):
            total += nums[j]
    return total


def check_all(nums, expected, label):
    for key, fn in impls.items():
        actual = fn(list(nums))
        assert actual == expected, (label, key, nums, actual, expected)
    assert oracle(nums) == expected, (label, "oracle")
    print("PASS", label)


for idx, (nums, expected) in enumerate(examples, 1):
    check_all(nums, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

Các ví dụ "tự dựng" dưới đây có kỳ vọng tính tay ngay trong comment.

In [23]:
check_all([5], 5, "một phần tử: start = max(0, -5) = 0 -> [5]")
check_all([1, 1], 1 + 2, "i=0: [1]; i=1: start = 0 -> [1,1] = 2")
check_all([1, 1, 1, 1], 1 + 2 + 2 + 2, "mỗi i >= 1 chỉ lùi 1 ô -> hai phần tử")
check_all([100, 100, 100], 100 + 200 + 300, "giá trị lớn: luôn lấy từ đầu mảng")
check_all([1000] * 4, 1000 * (1 + 2 + 3 + 4), "1000 > mọi chỉ số: luôn start = 0")
# tự dựng [1,2,3]: i=0 -> [1]=1 ; i=1: start=max(0,1-2)=0 -> [1,2]=3 ; i=2: start=max(0,2-3)=0 -> [1,2,3]=6 ; tổng 10
check_all([1, 2, 3], 1 + 3 + 6, "[1,2,3]: 1 + 3 + 6")
# tự dựng [2,1,1,1]: i=0 -> [2]=2 ; i=1: start=0 -> [2,1]=3 ; i=2: start=1 -> [1,1]=2 ; i=3: start=2 -> [1,1]=2 ; tổng 9
check_all([2, 1, 1, 1], 2 + 3 + 2 + 2, "[2,1,1,1]: 2 + 3 + 2 + 2")

PASS một phần tử: start = max(0, -5) = 0 -> [5]
PASS i=0: [1]; i=1: start = 0 -> [1,1] = 2
PASS mỗi i >= 1 chỉ lùi 1 ô -> hai phần tử
PASS giá trị lớn: luôn lấy từ đầu mảng
PASS 1000 > mọi chỉ số: luôn start = 0
PASS [1,2,3]: 1 + 3 + 6
PASS [2,1,1,1]: 2 + 3 + 2 + 2


### 12.2. Phản ví dụ cho các cách làm sai

Hai cách sai thường gặp: (a) **quên `max(0, ...)`** và để chỉ số âm lọt vào phép cắt lát; (b) **cắt lát thiếu phần tử `i`** (`nums[start:i]`).

In [24]:
def wrong_no_clamp(nums):
    return sum(sum(nums[i - x: i + 1]) for i, x in enumerate(nums))          # SAI: i - x có thể âm


def wrong_exclusive_end(nums):
    return sum(sum(nums[max(0, i - x): i]) for i, x in enumerate(nums))      # SAI: thiếu nums[i]


print(wrong_no_clamp([2, 3, 1]), "!=", sol.subarraySum([2, 3, 1]))
print(wrong_exclusive_end([2, 3, 1]), "!=", sol.subarraySum([2, 3, 1]))
assert wrong_no_clamp([2, 3, 1]) != sol.subarraySum([2, 3, 1])
assert wrong_exclusive_end([2, 3, 1]) != sol.subarraySum([2, 3, 1])

7 != 11
5 != 11


### 12.3. Đối chiếu ngẫu nhiên (có seed) và tính chất

Sinh `nums` ngẫu nhiên với `n` trong `[1, 100]`, giá trị trong `[1, 1000]` (đúng ràng buộc). Kiểm tra: (1) mọi cách + `oracle` cùng kết quả; (2) tính chất: tổng `>= sum(nums)` (mỗi mảng con chứa ít nhất `nums[i]`, nên ≥ tổng các `nums[i]`) và `<= n * sum(nums)`; (3) nếu mọi `nums[i] >= n` thì mỗi mảng con là cả đoạn `nums[0..i]`, tổng bằng tổng các tiền tố.

In [25]:
rng = random.Random(3427)
for trial in range(2000):
    n = rng.randint(1, 100)
    nums = [rng.randint(1, 1000) for _ in range(n)]
    ref = oracle(nums)
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
    assert sum(nums) <= ref <= n * sum(nums)
print("PASS 2000 mảng ngẫu nhiên đúng ràng buộc")

# Tính chất (3): nums[i] >= n -> mọi mảng con bắt đầu từ 0
for trial in range(300):
    n = rng.randint(1, 100)
    nums = [rng.randint(n, 1000) for _ in range(n)]
    assert sol.subarraySum(nums) == sum(accumulate(nums))     # tổng các tổng tiền tố
print("PASS 300 mảng với mọi nums[i] >= n")

PASS 2000 mảng ngẫu nhiên đúng ràng buộc
PASS 300 mảng với mọi nums[i] >= n


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên `max(0, ...)` — chỉ số âm trong cắt lát

In [26]:
def sai_quen_chan_bien(nums):
    total = 0
    for i, x in enumerate(nums):
        start = i - x                              # SAI: có thể âm
        total += sum(nums[start:i + 1])
    return total


print("sai:", sai_quen_chan_bien([2, 3, 1]), "| đúng:", Solution().subarraySum([2, 3, 1]))
print("nums[-2:1] =", [2, 3, 1][-2:1], "-> rỗng, nên i=0 đóng góp 0 thay vì 2")

sai: 7 | đúng: 11
nums[-2:1] = [] -> rỗng, nên i=0 đóng góp 0 thay vì 2


**Vì sao sai:** ở `i = 0`, `start = 0 - 2 = -2`; `nums[-2:1]` nghĩa là "từ chỉ số `1` (thứ hai từ cuối) đến chỉ số `0`" — một đoạn **rỗng** — nên mất đóng góp `2` mà Python **không báo lỗi**. **Cách sửa:** `start = max(0, i - x)`.

### Lỗi 2: Lệch một khi dùng tiền tố (`s[i]` thay vì `s[i + 1]`)

In [27]:
def sai_lech_mot(nums):
    s = list(accumulate(nums, initial=0))
    return sum(s[i] - s[max(0, i - x)] for i, x in enumerate(nums))          # SAI: s[i] là tổng đến i - 1


print("sai:", sai_lech_mot([2, 3, 1]), "| đúng:", Solution().subarraySum([2, 3, 1]))

sai: 5 | đúng: 11


**Vì sao sai:** `s[i]` là tổng `i` phần tử đầu, tức đến chỉ số `i - 1`; vậy `s[i] - s[start]` thiếu `nums[i]`. Muốn **gồm cả `nums[i]`** phải dùng `s[i + 1]`. **Cách sửa:** nhớ công thức `s[r + 1] - s[l]` cho đoạn `l..r` gồm cả hai đầu.

### Lỗi 3: Dựng tiền tố không có số `0` ở đầu

In [28]:
def sai_thieu_so_0(nums):
    s = list(accumulate(nums))                     # SAI: thiếu initial=0, s[k] giờ là tổng đến chỉ số k
    return sum(s[i] - s[max(0, i - x) - 1] for i, x in enumerate(nums))      # start = 0 -> s[-1] là PHẦN TỬ CUỐI


print("s thiếu số 0:", list(accumulate([2, 3, 1])))
print("sai:", sai_thieu_so_0([2, 3, 1]), "| đúng:", Solution().subarraySum([2, 3, 1]))

s thiếu số 0: [2, 5, 6]
sai: -1 | đúng: 11


**Vì sao sai:** không có `s[0] = 0`, đoạn bắt đầu từ `0` phải xử lý riêng; nếu không, `s[-1]` (phần tử cuối) bị trừ nhầm. **Cách sửa:** luôn dựng `s` dài `n + 1` với `s[0] = 0` (`accumulate(nums, initial=0)`) để mọi đoạn dùng chung một công thức.

### Lỗi 4: Cắt lát thiếu phần tử `i`

In [29]:
def sai_thieu_i(nums):
    return sum(sum(nums[max(0, i - x):i]) for i, x in enumerate(nums))      # SAI: nums[a:i] không gồm i


print("sai:", sai_thieu_i([2, 3, 1]), "| đúng:", Solution().subarraySum([2, 3, 1]))

sai: 5 | đúng: 11


**Vì sao sai:** cắt lát nửa mở `nums[a:b]` không gồm `b`; đề muốn gồm `nums[i]` nên phải là `nums[a : i + 1]`. **Cách sửa:** `i + 1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Tổng của một đoạn liên tục**, nhiều lần" ⇒ **tổng tiền tố**: `s[r + 1] - s[l]` cho đoạn `l..r`.
- Luôn dựng `s` với phần tử đầu `s[0] = 0` (`accumulate(nums, initial=0)`) để mọi đoạn dùng cùng một công thức, kể cả đoạn bắt đầu từ `0`.
- "**Biên phụ thuộc giá trị**" như `max(0, i - nums[i])` ⇒ chặn biên bằng `max`/`min` thay vì `if`, và **nhớ rằng chỉ số âm của Python không báo lỗi**.
- Đoạn bài "cộng `+1` lên nhiều đoạn" ⇒ **mảng hiệu** (`diff[l] += 1; diff[r + 1] -= 1`), rồi `accumulate`.
- Khi ràng buộc nhỏ (`n <= 100`) cách `O(n²)` vẫn qua, nhưng hãy luyện viết cách `O(n)` bằng tiền tố để quen kỹ thuật.
- Tự kiểm tra: tổng cuối luôn `>= sum(nums)` (mỗi mảng con chứa ít nhất chính `nums[i]`).

## 15. Biến thể của bài

1. **Cho nhiều truy vấn tổng đoạn** `(l, r)` rồi trả lời mỗi truy vấn bằng tiền tố (LeetCode 303 Range Sum Query - Immutable).
2. **Mảng con kết thúc ở `i` và lùi đúng `k` bước** (cửa sổ cố định): `s[i + 1] - s[max(0, i + 1 - k)]`.
3. **Mảng con bắt đầu ở `i` và vươn sang phải `nums[i]` bước**: `end = min(n - 1, i + nums[i])`.
4. **Đếm số lần mỗi phần tử được tính** (mảng `cover`) — chính là Cách 5.
5. **Tổng tiền tố trên ma trận 2D** (LeetCode 304 Range Sum Query 2D) hoặc **Subarray Sum Equals K** (LeetCode 560).

In [30]:
def sum_window_k(nums, k):
    """Biến thể 2: mảng con kết thúc ở i, độ dài tối đa k."""
    s = list(accumulate(nums, initial=0))
    return sum(s[i + 1] - s[max(0, i + 1 - k)] for i in range(len(nums)))


def sum_forward(nums):
    """Biến thể 3: mảng con bắt đầu ở i, vươn sang phải nums[i] bước."""
    s = list(accumulate(nums, initial=0))
    n = len(nums)
    return sum(s[min(n - 1, i + x) + 1] - s[i] for i, x in enumerate(nums))


def coverage(nums):
    """Biến thể 4: mỗi phần tử nằm trong bao nhiêu mảng con của đề."""
    n = len(nums)
    diff = [0] * (n + 1)
    for i, x in enumerate(nums):
        diff[max(0, i - x)] += 1
        diff[i + 1] -= 1
    return list(accumulate(diff))[:n]


nums = [2, 3, 1]
print(sum_window_k(nums, 2))        # [2] + [2,3] + [3,1] = 2 + 5 + 4 = 11
print(sum_forward(nums))            # i=0: [2,3,1]=6 ; i=1: [3,1]=4 ; i=2: [1]=1 -> 11
print(coverage(nums))               # 2 nằm trong 2 mảng con, 3 trong 2, 1 trong 1 -> [2, 2, 1]
assert sum_window_k(nums, 2) == 11
assert sum_forward(nums) == 11
assert coverage(nums) == [2, 2, 1]
assert sum(a * b for a, b in zip(nums, coverage(nums))) == Solution().subarraySum(nums)

11
11
[2, 2, 1]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp trong | cộng từng phần tử mỗi mảng con | `O(n²)` | `O(1)` | `n` nhỏ, bám sát đề, làm trọng tài |
| 2. Cắt lát + `sum` | `sum(nums[start:i+1])` | `O(n²)` | `O(n)` | viết nhanh khi `n <= 100` |
| 3. Tiền tố tự viết | `s[i+1] - s[start]` | `O(n)` | `O(n)` | **nên nộp**; dễ hiểu cho người mới |
| 4. `accumulate(initial=0)` | tiền tố một dòng | `O(n)` | `O(n)` | gọn nhất, đúng lời giải mẫu |
| 5. Mảng hiệu | đếm số lần phủ `cover[j]` | `O(n)` | `O(n)` | khi cần biết mỗi phần tử được tính mấy lần |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `list(accumulate([1, 2, 3], initial=0))`, `max(0, -4)`, `[5, 6, 7, 8][1:3]`, `Solution().subarraySum([1])`, `Solution().subarraySum([1, 1])`.

**Bài 2 (tính tay).** Với `nums = [1, 3, 2, 1]`: tính `start` cho từng `i`, liệt kê mảng con, tổng từng mảng con và tổng cuối.

**Bài 3 (điền TODO).** Hoàn thành `subarray_sum_todo(nums)` theo Cách 3 (tiền tố tự viết).

In [31]:
def subarray_sum_todo(nums):
    # TODO 1: s = [0]; với mỗi x trong nums: s.append(s[-1] + x)
    total = 0
    # TODO 2: for i, x in enumerate(nums): start = max(0, i - x)
    # TODO 3: total += s[i + 1] - s[start]
    # TODO 4: return total
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `subarray_sum_from_scratch(nums)` theo **Cách 5** (mảng hiệu) **mà không dùng** `itertools.accumulate`: tự cộng dồn `diff` bằng một biến `running`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `range_sum_queries(nums, queries)` nhận danh sách truy vấn `(l, r)` và trả về tổng `nums[l..r]` cho từng truy vấn bằng tiền tố. (b) Viết `max_single_subarray_sum(nums)` trả về **tổng lớn nhất** trong các mảng con của đề (chỉ một mảng con, không cộng dồn các mảng).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[0, 1, 3, 6]`, `0`, `[6, 7]`, `1`, `3`. Bài 3: dựng `s` trước, rồi dùng `s[i + 1] - s[start]`.
- **Mức 2 (vừa):** Bài 2: `i=0: start=0 → [1]=1`; `i=1: start=max(0,1-3)=0 → [1,3]=4`; `i=2: start=0 → [1,3,2]=6`; `i=3: start=2 → [2,1]=3`; tổng `14`. Bài 4: `running += diff[j]` rồi dùng `running` làm `cover[j]`.
- **Mức 3 (sâu):** Bài 5(a): `s = [0]` dựng bằng vòng lặp; đáp án `s[r + 1] - s[l]`. 5(b): `max(s[i + 1] - s[max(0, i - x)] for i, x in enumerate(nums))`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert list(accumulate([1, 2, 3], initial=0)) == [0, 1, 3, 6]
assert max(0, -4) == 0
assert [5, 6, 7, 8][1:3] == [6, 7]
assert Solution().subarraySum([1]) == 1
assert Solution().subarraySum([1, 1]) == 3

# Bài 2
assert Solution().subarraySum([1, 3, 2, 1]) == 1 + 4 + 6 + 3 == 14


# Bài 3
def subarray_sum_todo_ans(nums):
    s = [0]
    for x in nums:
        s.append(s[-1] + x)
    total = 0
    for i, x in enumerate(nums):
        start = max(0, i - x)
        total += s[i + 1] - s[start]
    return total


for nums, expected in examples:
    assert subarray_sum_todo_ans(nums) == expected


# Bài 4
def subarray_sum_from_scratch(nums):
    n = len(nums)
    diff = [0] * (n + 1)
    for i, x in enumerate(nums):
        diff[max(0, i - x)] += 1
        diff[i + 1] -= 1
    running = 0
    total = 0
    for j in range(n):
        running += diff[j]                 # running chính là cover[j]
        total += nums[j] * running
    return total


for nums, expected in examples:
    assert subarray_sum_from_scratch(nums) == expected


# Bài 5
def range_sum_queries(nums, queries):
    s = [0]
    for x in nums:
        s.append(s[-1] + x)
    return [s[r + 1] - s[l] for l, r in queries]


def max_single_subarray_sum(nums):
    s = list(accumulate(nums, initial=0))
    return max(s[i + 1] - s[max(0, i - x)] for i, x in enumerate(nums))


assert range_sum_queries([2, 3, 1], [(0, 0), (0, 2), (1, 2)]) == [2, 6, 4]
assert max_single_subarray_sum([3, 1, 1, 2]) == 4
print("Tất cả đáp án đều đúng")

Tất cả đáp án đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tổng nhiều đoạn bằng tiền tố `s` dài `n + 1`)

```python
class Solution:
    def subarraySum(self, nums: List[int]) -> int:
        s = list(accumulate(nums, initial=0))          # s[k] = tổng k phần tử đầu, s[0] = 0
        return sum(s[i + 1] - s[max(0, i - x)] for i, x in enumerate(nums))
```

**Checklist**

- [ ] `s` có độ dài `n + 1` và `s[0] = 0` (`initial=0`)?
- [ ] Đoạn `l..r` gồm cả hai đầu dùng `s[r + 1] - s[l]` (không phải `s[r] - s[l]`)?
- [ ] `start = max(0, i - nums[i])` đã chặn biên (không để chỉ số âm lọt vào cắt lát/tiền tố)?
- [ ] Nếu dùng cắt lát, đã nhớ `nums[start : i + 1]` (cộng 1 vì nửa mở)?
- [ ] Đã thử `n = 1`, mọi `nums[i] >= n` (luôn bắt đầu từ `0`) và `nums[i] = 1` (mảng con dài 2)?
- [ ] Đã so với bản vét cạn trên nhiều input ngẫu nhiên?